<a href="https://colab.research.google.com/github/MatikMo/FSDTask/blob/main/LSTM/FSDTask_LSTM_BCE.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# LSTM Model to classify working/broken breaks

First of all we need some dependencies:

In [2]:
#Dependencies, comment out install comments fpr use with GPU

#!pip install comet_ml > /dev/null 2>&1
#import comet_ml
#COMET_API_KEY = "jbaPpXFZ6mRVGWSatsxpOSi7v"
#assert torch.cuda.is_available(), "Please enable GPU from runtime settings"

# Import PyTorch and other relevant libraries
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset, Sampler, random_split

#To use the periodic plotter
#!pip install mitdeeplearning --quiet
#import mitdeeplearning as mdl

#Numpy, Matplotlib and random
import numpy as np
import numpy.random as rd
import matplotlib.pyplot as plt
import random

#Nice Visualisation
from tqdm import tqdm

#Python OS module
import os

#Pandas to extract Data
import pandas as pd

#For preprocessing the date
from sklearn import preprocessing as pp
from sklearn.model_selection import train_test_split

As next step, we import the Data. For now we only use the sensor measurements to train the LSTM.

For the training labels, we define 0/False as a working break and 1/True as a broken break. 

We try out the option of normalizing measurement data by using the maxAbs Method. So all data gets devided my the maxAbs value (no shifting along the y-axis).


In [3]:
#Function to convert labels into 0/1 for broken/working breaks
def label_conversion(label_raw):
  label = (label_raw != 0).to(dtype=torch.float32)
  return label

#To normalize the measuremens with its maximal absolute ax-data-point per measurement
def norm(sensor_data_raw):
  data_max = torch.amax(torch.abs(sensor_data_raw), dim=1, keepdim=True)
  ax_max = data_max[:, :, 0].unsqueeze(-1)
  sensor_data = sensor_data_raw / ax_max
  return sensor_data

Now we import the data and apply normalization if we want to

In [4]:
#Data to train and test
df = pd.read_pickle('../../data/train.pickle')

#import Data
sensor_data_raw = torch.tensor(df['sensor_data'],dtype=torch.float32)
label_raw = torch.tensor(df['label'])

#using label conversion and normalisation
label = label_conversion(label_raw)
sensor_data = sensor_data_raw #normalize(sensor_data_raw)

/tmp/ipykernel_1500/2661493813.py:5: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /pytorch/torch/csrc/utils/tensor_new.cpp:254.)
  sensor_data_raw = torch.tensor(df['sensor_data'],dtype=torch.float32)


## Data Generation

The following cells are optional for hard coded data extension. 

This happens by shifting the existing data in time (quadraticly extrapolate new points) and scale by a random factor

In [5]:
#To generate more data, we take an existing measurements, scale them slightly in different ways per measurements ax,ay,az,rx,ry,rz
#or shift them as total in time
def generate_data(measurement):
  #varianz
  max_shift = 5 #timesteps back/forward
  min_scale = 0.95
  max_scale = 1.05
  #get time shift
  m_len = measurement.size(dim=0)
  n_measurements = measurement.size(dim=1)
  shift_range = np.concatenate([np.arange(-max_shift,-1),np.arange(1,max_shift)])
  time_shift = random.choice(shift_range)
  shifted_measurement = torch.zeros_like(measurement)

  #shift data and interpolate
  if time_shift < 0:
    shifted_measurement[0:m_len+time_shift] = measurement[-time_shift:m_len]
    #go trough measurements and interpolate some data
    for i in range(n_measurements):
      x = np.array([m_len+time_shift-3, m_len+time_shift-2, m_len+time_shift-1])
      y = np.array([shifted_measurement[x[0]][i],shifted_measurement[x[1]][i],shifted_measurement[x[2]][i]])
      coeffs = np.polyfit(x,y,deg=2)
      f = np.poly1d(coeffs)
      for j in range(m_len+time_shift,m_len):
        shifted_measurement[j][i] = f(j)

  if time_shift > 0:
    shifted_measurement[time_shift:m_len] = measurement[0:m_len-time_shift]
    #go trough measurements and interpolate some data
    for i in range(n_measurements):
      x = np.array([time_shift, time_shift+1, time_shift+2])
      y = np.array([shifted_measurement[x[0]][i],shifted_measurement[x[1]][i],shifted_measurement[x[2]][i]])
      coeffs = np.polyfit(x,y,deg=2)
      f = np.poly1d(coeffs)
      for j in range(0,time_shift):
        shifted_measurement[j][i] = f(j)

  #scale measurements somehow
  shape = (1,6)
  scalars = (max_scale - min_scale) * torch.rand(shape) + min_scale
  shifted_and_scaled = shifted_measurement * scalars

  #test
  #for i,j in [[10,0],[14,1],[20,2],[30,3],[40,4],[50,5]]:
    #print(shifted_and_scaled[i+time_shift][j].item()/measurement[i][j].item())

  return shifted_and_scaled

Loop to increase data

In [8]:
#we increase the amount of data by a factor
factor = 4
sensor_data_aug = [sensor_data_raw]
label_aug = [label]

for id, el in enumerate(sensor_data_raw):
    new_samples = []
    new_labels = []
    for _ in range(factor):
        new_data = generate_data(el)
        new_samples.append(new_data.unsqueeze(0))  # (1, 128, 6)
        new_labels.append(label[id].unsqueeze(0))  # (1,)
    sensor_data_aug.append(torch.cat(new_samples, dim=0))
    label_aug.append(torch.cat(new_labels, dim=0))

# Concanate all Parts
sensor_data_full = torch.cat(sensor_data_aug, dim=0)
label_full = torch.cat(label_aug, dim=0)

print(sensor_data_full.shape)
print(np.bincount(label))

#Save new Dataset
torch.save({
    'features': sensor_data_full,
    'labels': label_full
}, '../../data/augmented_dataset.pt')


torch.Size([20265, 128, 6])
[1798 2255]


Loading saved data

In [ ]:
#Load augmented data
data = torch.load('../../data/augmented_dataset.pt')
sensor_data = data['features']
label = data['labels']
print(np.bincount(label))


[ 8990 11275]


## Data Loader and Model

As next step we need helper functions to get batches out of the data.


In [9]:
#split data into validation data and training data and generate data loaders
def split_data(sensor_data, label, rand_seed):
  #use build in test_split method
  sensor_data_train, sensor_data_val, label_train, label_val = train_test_split(
    sensor_data,
    label,
    test_size = 0.25,
    shuffle=True,
    random_state=rand_seed
  )
  #concernate data again to build data loader
  train_data = TensorDataset(sensor_data_train, label_train)
  val_data = TensorDataset(sensor_data_val, label_val)
  return train_data, val_data

def get_loader(train_data, val_data, batch_size):
  #dataloader
  train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=True)
  val_loader = DataLoader(val_data, batch_size=batch_size, shuffle=True)

  return train_loader, val_loader


Now we can define our LSTM Modell such that we input the measurement-batches and get for every sequence in the batch a logit 0/1 for working/broken breaks

In [10]:
### Defining the LSTM Model ###
class LSTMModel(nn.Module):
    def __init__(self, input_dim, hidden_size, batch_size, num_layers, drobout):
        super(LSTMModel, self).__init__()
        # Dimension of hidden layer
        self.hidden_size = hidden_size
        #generate LSTM Network
        self.lstm = nn.LSTM(input_dim, hidden_size, num_layers=num_layers, dropout=dropout, batch_first=True)
        #Linear Dense Layer
        self.fc = nn.Linear(hidden_size,1) #changed for another label-handling

    #Same as in coding lab to initialize LSTM
    def init_hidden(self, batch_size, device):
        # Initialize hidden state and cell state with zeros
        return (torch.zeros(num_layers, batch_size, self.hidden_size).to(device),
                torch.zeros(num_layers, batch_size, self.hidden_size).to(device))

    def forward(self, x, state=None, return_state=False):

        if state is None:
            state = self.init_hidden(x.size(0), x.device)
        out, state = self.lstm(x, state)
        #nur letztes label zählt
        out = self.fc(out[:,-1,:])
        return out if not return_state else (out, state)


Now its already time to think about the parameters, the optimizer and the loss function to train out network.

Since we have biased data, we use weights on our BCEWithLogitsLoss function

In [12]:
#Intialize modell
##Measurements have 6 datapoints per timestep
input_dim = sensor_data.size(dim=2)
##Length of one Measurement
seq_length = sensor_data.size(dim=1)
##Number of hidden neurons
hidden_size = 512
##Number of measuement per batch
batch_size = 64
##Training Iterations and
num_epochs = 3*20
#Checkpoint Infrastructure
checkpoint_dir = 'Models/training_checkpoints'
os.makedirs(checkpoint_dir, exist_ok=True)
training_attempt = 1002

#Amount of layers
num_layers = 1
#Dropout
dropout = 0.5

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = LSTMModel(input_dim, hidden_size, batch_size, num_layers, dropout).to(device)
# model.load_state_dict(torch.load('/content/drive/MyDrive/training_checkpoints/modell8_best', weights_only=False))
optimizer = torch.optim.Adam(model.parameters(), lr=5e-2, weight_decay=1e-5)

# # Test the model with some sample data
# x, y = get_batch(sensor_data_train, label_train, batch_size)
# x = x.to(device)
# y = y.to(device)

# pred = model(x)
# #print("Label shape:      ", y.shape, " # (batch_size, sequence_length)")
# #print("Prediction shape: ", pred.shape, "# (batch_size, sequence_length, vocab_size)")


#There are around 40% of broken breaks and 60% of working breaks in the data
#We try to tackle this problem in a first approach by giving weights to the loss function


weight = np.bincount(label)[0].item() / np.bincount(label)[1].item()
print(np.bincount(label)[1].item())
print(weight)
weight = torch.tensor(weight)
loss_fn = nn.BCEWithLogitsLoss(pos_weight=1/weight)



/home/maurix/anaconda3/lib/python3.11/site-packages/torch/nn/modules/rnn.py:123: UserWarning: dropout option adds dropout after all but last recurrent layer, so non-zero dropout expects num_layers greater than 1, but got dropout=0.5 and num_layers=1
  warnings.warn(


2255
0.797339246119734


Now we define a train and an evaluate function 

In [13]:
def train_step(x, y):
  # Set the model's mode to train
  model.train()
  # Zero gradients for every step
  optimizer.zero_grad()
  # forward and compute loss
  y_hat = model(x)
  loss = loss_fn(y_hat.squeeze(),y.float())
  #gradient computation and optimization
  loss.backward()
  #gradient clipping against exploding gadients
  torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
  #step go right, step to left one to the front one to the back, turn the hipps, clap the hands 7 x 7 is finer sand (s)
  optimizer.step()

  return loss

def evaluate(val_loader, set_max, test_eval=False):
    #maximal amount of batches drawn
    max_batches = round(100/batch_size)+1
    model.eval()
    correct, total = 0, 0
    pred1label0 = 0
    pred0label1 = 0
    label0 = 0
    label1 = 0
    with torch.no_grad():
      for xv, yv in val_loader:

        max_batches = max_batches - 1
        if max_batches < 0 and set_max:
          break

        xv = xv.to(device)
        yv = yv.to(device)

        y_hat = model(xv)
        probs = torch.sigmoid(y_hat)
        pred = (probs > 0.5).long()
        yv = yv.long()
        total += yv.size(0)

        pred = pred.squeeze()
        yv = yv.squeeze()

        correct += (pred == yv).sum().item()

        pred1label0 += (pred-1 == yv).sum().item()
        pred0label1 += (pred+1 == yv).sum().item()

        label0 += (0 == yv).sum().item()
        label1 += (1 == yv).sum().item()

        #print("Sample Pred and Label:")
        #print(pred)
        #print(yv)

    if test_eval:
      print(f"Predicted 0, but brake-label 1 given:{pred0label1}, acc0.:{round(1 - (pred1label0/label0),3)}")
      print(f"Predicted 1, but brake-label 0 given:{pred1label0}, acc1.:{round(1 - (pred0label1/label1),3)}")

    return correct/total


## Training Loop

In [ ]:
#Move model to GPU and train
model.to(device)

history = []
#plotter = mdl.util.PeriodicPlotter(sec=2, xlabel='Iterations', ylabel='Loss')

#initial loader
train_data, val_data = split_data(sensor_data, label, 161)
train_loader, val_loader = get_loader(train_data, val_data, batch_size)

if hasattr(tqdm, '_instances'): tqdm._instances.clear() # clear if it exists
for epoch in range(num_epochs):

    print(f"In Epoch {epoch}/{num_epochs}")

    #count iteration per epoch
    iter = 0
    acc = 0

    #split into new train and val data after 3 epochs
    if epoch % 3 == 0:
      seed = pow(epoch, 7) % 419
      train_data, val_data = split_data(sensor_data, label, seed)
      train_loader, val_loader = get_loader(train_data, val_data, batch_size)

    progress_bar = tqdm(train_loader, desc=f"Progress in Epoch {epoch}", unit="Batch")
    for x,y in progress_bar:

      # Convert numpy arrays to PyTorch tensors
      x_batch = torch.tensor(x, dtype=torch.float32).to(device)
      y_batch = torch.tensor(y, dtype=torch.float32).to(device)

      # Take a train step
      loss = train_step(x_batch, y_batch)

      #to visualize within the plotter
      history.append(loss.item())
      #plotter.plot(history)

      # Print progress
      if iter % 20 == 0:
          acc = evaluate(val_loader, set_max=True, test_eval=False)
      #update progress bar
      progress_bar.set_postfix(accuracy=f"{acc:.2f}")

      #increase iteration index
      iter = iter + 1

    #save modell after each epoch
    checkpoint_prefix = os.path.join(checkpoint_dir, "modell" + str(training_attempt) + "epoch" + str(epoch) + "acc" + str(round(acc,2)))
    torch.save(model.state_dict(), checkpoint_prefix)

    #shuffle train/val data new after each epoch
    train_loader, val_loader = get_loader(train_data, val_data, batch_size)

# Save the final trained model
torch.save(model.state_dict(), checkpoint_prefix)

## Testing

Here we load the test data

In [16]:
#Data to train
df.test = pd.read_pickle('../../data/test.pickle')

#import Data
sensor_data_test_raw = torch.tensor(df.test['sensor_data'],dtype=torch.float32)
label_test_raw = torch.tensor(df.test['label'])

#conversion and normalisation as above
label_test = label_conversion(label_test_raw)
sensor_data_test = sensor_data_test_raw #norm(sensor_data_test_raw)

#concernate and create data loader
test_data = TensorDataset(sensor_data_test, label_test)

/tmp/ipykernel_1500/1815101959.py:2: UserWarning: Pandas doesn't allow columns to be created via a new attribute name - see https://pandas.pydata.org/pandas-docs/stable/indexing.html#attribute-access
  df.test = pd.read_pickle('../../data/test.pickle')


And test it on one or more models that we just trained

In [17]:
#Loading different training modells
# 
# constant parameters:
#  - Num Layers  ---------- 2
#  - Layer Size ----------- 512
#  - Epochs --------------- 20
#  - Iteration per Epoch -- 3
#
# Model           
# -------------------------------------------------------------------------------------------------------------
# Batch Size
# Learning Rate
# 

hidden_size = 512
batch_size = 32
num_layers = 1
dropout = 0.3

#Fraction of Validation Data: 0.25


test_loader = DataLoader(test_data, batch_size=batch_size)

file_path = 'Models/LSTM_hardcodedaug' #or 'Models/training_checkpoints/NAME_OF_NEW_TRAINED_NETWORK'
print('Accuracy for ' + file_path)
model = LSTMModel(input_dim, hidden_size, batch_size, num_layers, dropout).to(device)
model.load_state_dict(torch.load(file_path, map_location=torch.device('cpu')))


#evaluate actual effeciency on test data
print(evaluate(test_loader, set_max=False, test_eval=True))


Accuracy for Models/LSTM_hardcodedaug


/home/maurix/anaconda3/lib/python3.11/site-packages/torch/nn/modules/rnn.py:123: UserWarning: dropout option adds dropout after all but last recurrent layer, so non-zero dropout expects num_layers greater than 1, but got dropout=0.3 and num_layers=1
  warnings.warn(


Predicted 0, but brake-label 1 given:233, acc0.:0.872
Predicted 1, but brake-label 0 given:76, acc1.:0.429
0.6913086913086913


Only Run if you want to clear the training_checkpoints folder

In [ ]:
# import shutil
# import os

# folder_path = 'Models/training_checkpoints'

# # Prüfen, ob der Ordner existiert und dann löschen
# if os.path.exists(folder_path):
#     shutil.rmtree(folder_path)
#     print(f"Ordner '{folder_path}' wurde gelöscht.")
# else:
#     print(f"Ordner '{folder_path}' existiert nicht.")